# Practical Exam: Automating Customer Support with OpenAI API

You work as an AI Engineer at ChatSolveAI, a company that provides automated customer support solutions. The company wants to improve response times and accuracy in answering customer queries by leveraging OpenAI’s GPT models.

Your task is to build a chatbot that classifies customer queries, retrieves relevant responses, and logs interactions in a structured way. The chatbot will use text embeddings, similarity search, API calls, and conversation management techniques.


**Please note:** 

1. The OpenAI Embeddings API supports passing a list of strings to the input parameter in a single request. This allows you to generate multiple embeddings at once without looping over individual elements, which can significantly improve efficiency and reduce the risk of hitting rate limits.

2. When submitting your solution, you may see an error message reading 'Something went wrong while submitting your solution. Please try again.' This is because using the OpenAI API may mean code takes longer to run than code in our other Certifications. Please ignore this message if your code is taking a few minutes to run. However, if your code makes too many API requests, the API will time out. If your cells run for more than a few minutes each, you may need to consider revising your code. 

In [6]:
# Run this cell before running your solution

# Import necessary modules
import os
from openai import OpenAI

# Define the model to use
model = "gpt-3.5-turbo"

# Define the client
client = OpenAI()

# Task 1

ChatSolveAI has provided a knowledge base (`knowledge_base.csv`) containing information about various products, services, and customer policies. To enhance search and query capabilities, you need to convert this data into embeddings and store them for efficient retrieval.

- Load the dataset (`knowledge_base.csv`).
- Generate text embeddings using OpenAI’s embedding model (`text-embedding-3-small`). Each document's `document_text` should be transformed into an embedding vector. Do not apply any text transformations such as lowercasing, stripping or normalization before embedding.
- Store the generated embeddings in a structured format (`knowledge_embeddings.json`) with the following format available below.
- Store the embedded data and associated metadata for retrieval.  

### Format to store generated embeddings:
```json
[
    {
       "document_id": 1,
       "document_text": "Example document text.",
       "embedding_vector": [0.123, 0.456, ...],
       "metadata": "Additional document info"
    }
]
```

### Data description: 

| Column Name       | Criteria                                                |
|-------------------|---------------------------------------------------------|
| document_id       | Integer. Unique identifier for each document. No missing values. |
| document_text     | String. Text content of the knowledge base. Preprocessed and embedded. |
| embedding_vector  | List. Embedding representation of the `document_text`. |
| metadata          | String. Metadata for additional information. |


In [7]:
# Write your answer to Task 1 here
import os
import json
import pandas as pd
from openai import OpenAI

# Load the dataset
df = pd.read_csv("knowledge_base.csv")

# Initialize the OpenAI client
client = OpenAI()

# Generate embeddings for ALL document_text values in a single API request
# (passing a list — not looping — to improve efficiency and avoid rate limits)
texts = df["document_text"].tolist()

response = client.embeddings.create(
    input=texts,
    model="text-embedding-3-small"
)

# Build the structured list of embeddings
knowledge_embeddings = []
for i, row in df.iterrows():
    embedding_entry = {
        "document_id": int(row["document_id"]),
        "document_text": row["document_text"],
        "embedding_vector": response.data[i].embedding,
        "metadata": str(row["metadata"]) if "metadata" in row else ""
    }
    knowledge_embeddings.append(embedding_entry)

# Save to knowledge_embeddings.json
with open("knowledge_embeddings.json", "w") as f:
    json.dump(knowledge_embeddings, f)

print(f"✅ Saved {len(knowledge_embeddings)} embeddings to knowledge_embeddings.json")
print("Sample entry (truncated):")
sample = knowledge_embeddings[0].copy()
sample["embedding_vector"] = sample["embedding_vector"][:5]  # truncate for display
print(json.dumps(sample, indent=2))

✅ Saved 501 embeddings to knowledge_embeddings.json
Sample entry (truncated):
{
  "document_id": 1,
  "document_text": "This document provides details about payment methods. Customers should follow the outlined procedures to ensure compliance. Requires proof of purchase.",
  "embedding_vector": [
    0.004974365234375,
    0.01605224609375,
    0.02349853515625,
    0.01494598388671875,
    -0.01482391357421875
  ],
  "metadata": "Requires proof of purchase"
}


# Task 2

ChatSolveAI receives customer queries that need to be classified and matched with appropriate responses. Your task is to preprocess and embed these queries, perform similarity searches on predefined responses (contained in `predefined_responses.json`), and retrieve the most relevant responses.

- Load the dataset (`processed_queries.csv`).
- Retrieve responses by using cosine similarity to perform a similarity search against predefined responses in `predefined_responses.json`.
- Structure API requests properly and implement error handling, including retry mechanisms to handle rate limits.
- Format model responses as JSON to maintain consistency in output.
- Compute confidence scores for retrieved responses, scaled to 0-1.
- Store the structured responses in a JSON file (`query_responses.json`), suitable for integration with other applications. Your JSON file should be structured as follows:

| Column Name       | Criteria                                                   |
|-------------------|------------------------------------------------------------|
| query_id         | Integer. Unique identifier for each query. No missing values. |
| query_text       | String. Preprocessed query text. |
| top_responses    | List. Top 3 most relevant responses retrieved. |
| confidence_scores | List. Model-based confidence score for the top 3 responses. |

In [8]:
# Write your answer to Task 2 here
import os
import json
import time
import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI()

# ── 1. Load datasets ──────────────────────────────────────────────────────────
queries_df = pd.read_csv("processed_queries.csv")

with open("predefined_responses.json", "r") as f:
    predefined_responses = json.load(f)

# ── 2. Normalize predefined_responses to a flat list of strings ───────────────
if isinstance(predefined_responses, list):
    if len(predefined_responses) > 0 and isinstance(predefined_responses[0], dict):
        # e.g. [{"response_text": "..."}, ...]
        key = "response_text" if "response_text" in predefined_responses[0] else list(predefined_responses[0].keys())[0]
        predefined_texts = [r[key] for r in predefined_responses]
    else:
        # plain list of strings
        predefined_texts = [str(r) for r in predefined_responses]
elif isinstance(predefined_responses, dict):
    predefined_texts = list(predefined_responses.values())
else:
    predefined_texts = [str(predefined_responses)]

# ── 3. Helper: cosine similarity ──────────────────────────────────────────────
def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# ── 4. Helper: embed texts with retry on rate limit ───────────────────────────
def embed_texts(texts, model="text-embedding-3-small", retries=5):
    for attempt in range(retries):
        try:
            response = client.embeddings.create(input=texts, model=model)
            return [item.embedding for item in response.data]
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt
                print(f"Rate limit hit. Retrying in {wait}s…")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Embedding failed after maximum retries.")

# ── 5. Embed predefined responses (single batch request) ─────────────────────
predefined_embeddings = embed_texts(predefined_texts)

# ── 6. Embed all customer queries (single batch request) ──────────────────────
query_texts = queries_df["query_text"].tolist()
query_embeddings = embed_texts(query_texts)

# ── 7. Similarity search + confidence scores ──────────────────────────────────
results = []

for idx, row in queries_df.iterrows():
    q_embedding = query_embeddings[idx]

    similarities = [
        cosine_similarity(q_embedding, p_emb)
        for p_emb in predefined_embeddings
    ]

    top3_indices = np.argsort(similarities)[::-1][:3]
    top_responses = [predefined_texts[i] for i in top3_indices]

    sim_array = np.array(similarities)
    sim_min, sim_max = sim_array.min(), sim_array.max()
    if sim_max - sim_min > 0:
        scaled_scores = (sim_array - sim_min) / (sim_max - sim_min)
    else:
        scaled_scores = sim_array

    confidence_scores = [round(float(scaled_scores[i]), 4) for i in top3_indices]

    results.append({
        "query_id": int(row["query_id"]),
        "query_text": str(row["query_text"]),
        "top_responses": top_responses,
        "confidence_scores": confidence_scores
    })

# ── 8. Save to query_responses.json ───────────────────────────────────────────
with open("query_responses.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"✅ Saved {len(results)} query responses to query_responses.json")
print(json.dumps(results[0], indent=2))

✅ Saved 501 query responses to query_responses.json
{
  "query_id": 1,
  "query_text": "How can I contact customer support?",
  "top_responses": [
    "You can contact our customer support via email or live chat on our website.",
    "If you received a damaged product, please contact support with images for a replacement.",
    "Track your order by logging into your account and checking the 'Orders' section."
  ],
  "confidence_scores": [
    1.0,
    0.4766,
    0.4015
  ]
}


# Task 3

To provide seamless customer service, ChatSolveAI wants to develop a chatbot that can respond to customer queries efficiently by searching for relevant responses and generating new ones when necessary.

- Develop a chatbot that:
    - Accepts customer queries via text input.
    - Searches for the most relevant responses from a predefined set of responses (`chatbot_responses.json`).
    - Uses the OpenAI Embeddings API (`text-embedding-3-small`) to compute semantic similarity between queries.
    - If no relevant response is found from the predefined set, generates a new response using GPT-3.5-turbo.
- Stores conversation history, including:
    - Query text
    - Retrieved response
    - Timestamp of the interaction
    - Confidence score of the response
- Include one open-ended query not in the predefined responses (e.g., about the refund policy) to test the chatbot’s ability to handle unmatched queries.
- Include one paraphrased query about support hours (e.g., “When can I talk to someone from support?”) to test semantic similarity matching.
- Store structured chatbot responses in a JSON file (`sample_chatbot_responses.json`). Make sure they follow this format:
```json
[
    {
        "query_text": "How do I reset my password?",
        "retrieved_response": "You can reset your password by clicking 'Forgot Password' on the login page.",
        "timestamp": "2025-04-02T14:30:00Z",
        "confidence_score": 0.92
    },
    {
        "query_text": "What are your business hours?",
        "retrieved_response": "Our support team is available from 9 AM to 5 PM, Monday to Friday.",
        "timestamp": "2025-04-02T14:35:00Z",
        "confidence_score": 0.87
    }
]
```

In [9]:
# Write your answer to Task 3 here
import os
import json
import time
import numpy as np
from datetime import datetime, timezone
from openai import OpenAI

client = OpenAI()

# ── 1. Load chatbot_responses.json (normalize to list of strings) ─────────────
with open("chatbot_responses.json", "r") as f:
    raw_responses = json.load(f)

if isinstance(raw_responses, list):
    if len(raw_responses) > 0 and isinstance(raw_responses[0], dict):
        key = "response_text" if "response_text" in raw_responses[0] else list(raw_responses[0].keys())[0]
        predefined_texts = [r[key] for r in raw_responses]
    else:
        predefined_texts = [str(r) for r in raw_responses]
elif isinstance(raw_responses, dict):
    predefined_texts = list(raw_responses.values())
else:
    predefined_texts = [str(raw_responses)]

# ── 2. Helpers ────────────────────────────────────────────────────────────────
def cosine_similarity(vec_a, vec_b):
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed_texts(texts, model="text-embedding-3-small", retries=5):
    for attempt in range(retries):
        try:
            response = client.embeddings.create(input=texts, model=model)
            return [item.embedding for item in response.data]
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt
                print(f"Rate limit hit. Retrying in {wait}s…")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Embedding failed after maximum retries.")

def generate_response(query_text, retries=5):
    """Fallback: generate a response with GPT-3.5-turbo when no match found."""
    for attempt in range(retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You are a helpful customer support assistant."},
                    {"role": "user", "content": query_text}
                ],
                max_tokens=200
            )
            return completion.choices[0].message.content.strip()
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt
                print(f"Rate limit hit (GPT). Retrying in {wait}s…")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("GPT generation failed after maximum retries.")

# ── 3. Embed all predefined responses in one batch ───────────────────────────
print("Embedding predefined responses…")
predefined_embeddings = embed_texts(predefined_texts)

# ── 4. Define chatbot function ────────────────────────────────────────────────
SIMILARITY_THRESHOLD = 0.75  # below this → fallback to GPT

def chatbot_respond(query_text):
    # Embed the query
    q_embedding = embed_texts([query_text])[0]

    # Cosine similarity against all predefined responses
    similarities = [cosine_similarity(q_embedding, p) for p in predefined_embeddings]
    best_idx = int(np.argmax(similarities))
    best_score = similarities[best_idx]

    # Scale score to 0-1 using min-max
    sim_array = np.array(similarities)
    sim_min, sim_max = sim_array.min(), sim_array.max()
    if sim_max - sim_min > 0:
        scaled = (sim_array - sim_min) / (sim_max - sim_min)
    else:
        scaled = sim_array
    confidence = round(float(scaled[best_idx]), 4)

    if best_score >= SIMILARITY_THRESHOLD:
        retrieved = predefined_texts[best_idx]
    else:
        # No relevant predefined response → generate with GPT
        print(f"  ↳ No match found (score={best_score:.3f}). Generating with GPT…")
        retrieved = generate_response(query_text)
        confidence = round(best_score, 4)  # raw similarity as confidence

    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    return {
        "query_text": query_text,
        "retrieved_response": retrieved,
        "timestamp": timestamp,
        "confidence_score": confidence
    }

# ── 5. Run sample queries ─────────────────────────────────────────────────────
# Paraphrased query → should match via semantic similarity
query_support_hours = "When can I talk to someone from support?"

# Open-ended query NOT in predefined set → should trigger GPT fallback
query_refund = "What is your refund policy if I am not satisfied with the product?"

sample_queries = [
    query_support_hours,
    query_refund,
]

conversation_history = []

for query in sample_queries:
    print(f"\n🔍 Query: {query}")
    result = chatbot_respond(query)
    conversation_history.append(result)
    print(f"   Response: {result['retrieved_response'][:80]}…")
    print(f"   Confidence: {result['confidence_score']} | Time: {result['timestamp']}")

# ── 6. Save to sample_chatbot_responses.json ──────────────────────────────────
with open("sample_chatbot_responses.json", "w") as f:
    json.dump(conversation_history, f, indent=2)

print(f"\n✅ Saved {len(conversation_history)} interactions to sample_chatbot_responses.json")
print(json.dumps(conversation_history, indent=2))

Embedding predefined responses…

🔍 Query: When can I talk to someone from support?
  ↳ No match found (score=0.682). Generating with GPT…
   Response: Our support team is available 24/7 to assist you with any questions or issues yo…
   Confidence: 0.6821 | Time: 2026-05-23T07:25:51Z

🔍 Query: What is your refund policy if I am not satisfied with the product?
  ↳ No match found (score=0.561). Generating with GPT…
   Response: Our refund policy typically allows for a refund within a certain timeframe, usua…
   Confidence: 0.5608 | Time: 2026-05-23T07:25:53Z

✅ Saved 2 interactions to sample_chatbot_responses.json
[
  {
    "query_text": "When can I talk to someone from support?",
    "retrieved_response": "Our support team is available 24/7 to assist you with any questions or issues you may have. You can reach us via live chat, email, or phone at any time, and we will be happy to help you.",
    "timestamp": "2026-05-23T07:25:51Z",
    "confidence_score": 0.6821
  },
  {
    "query_text"